# Multiplanet Distribution Validation

Validate the output of `multiplanet_draw_planet_arrays.py` against expected distributions:
- **Planet counts**: Poisson from local Suzuki density
- **Mass/Semi-major axis**: Log-uniform draws
- **Period ratio**: Must be > 1.3 for 2-planet systems
- **Eccentricity**: Half-Gaussian (σ=0.3)
- **Inclination**: Gaussian scatter around 1000° (GULLS binary convention)
- **Longitudes**: Uniform in [0, 360)

In [ ]:
import glob
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Load all planet files
data_dir = "planets/test_multiplanet"
files = sorted(glob.glob(f"{data_dir}/*.planets.*.*"))
print(f"Found {len(files)} planet files")

# Combine all data
all_data = []
for f in files[:10]:  # Limit for quick testing
    d = np.loadtxt(f, skiprows=1)
    all_data.append(d)
    
data = np.vstack(all_data)
n_rows = len(data)
n_systems = n_rows // 3
print(f"Loaded {n_rows} rows ({n_systems} systems)")

# Reshape to (n_systems, 3, 7) - [system, slot, parameter]
data_3d = data.reshape(n_systems, 3, 7)
print(f"Shape: {data_3d.shape}")

# Column indices
MASS, SEMI_A, ECC, INC, OMEGA, LONG_NODE, ORBIT_TYPE = range(7)

## Planet Count Distribution

Count how many planets (OrbitType 1 or 2 with mass > 0) each system has.

In [ ]:
# Count planets per system (OrbitType 1 or 2 with mass > 0)
planets_mask = (data_3d[:, :2, MASS] > 0)  # Only first two slots (planets, not moon)
n_planets_per_system = planets_mask.sum(axis=1)

fig, ax = plt.subplots(figsize=(8, 5))
counts, bins, _ = ax.hist(n_planets_per_system, bins=[-0.5, 0.5, 1.5, 2.5], 
                          edgecolor='black', alpha=0.7)
ax.set_xlabel("Number of Planets per System")
ax.set_ylabel("Count")
ax.set_title("Planet Count Distribution")
ax.set_xticks([0, 1, 2])

# Annotate with percentages
for i, c in enumerate(counts):
    pct = 100 * c / n_systems
    ax.annotate(f'{int(c)} ({pct:.1f}%)', xy=(i, c), ha='center', va='bottom', fontsize=10)

# Summary stats
print(f"Systems with 0 planets: {(n_planets_per_system == 0).sum()} ({100*(n_planets_per_system == 0).mean():.1f}%)")
print(f"Systems with 1 planet:  {(n_planets_per_system == 1).sum()} ({100*(n_planets_per_system == 1).mean():.1f}%)")
print(f"Systems with 2 planets: {(n_planets_per_system == 2).sum()} ({100*(n_planets_per_system == 2).mean():.1f}%)")
print(f"Mean planets/system: {n_planets_per_system.mean():.3f}")
plt.tight_layout()
plt.show()

## Mass and Semi-Major Axis Distributions

Should be log-uniform over their respective ranges.

In [ ]:
# Extract all planets with mass > 0
all_planets = data[data[:, MASS] > 0]
print(f"Total planets/moons: {len(all_planets)}")

# Separate planets from moons
planets_only = all_planets[all_planets[:, ORBIT_TYPE] <= 2]
moons_only = all_planets[all_planets[:, ORBIT_TYPE] == 3]
print(f"Planets: {len(planets_only)}, Moons: {len(moons_only)}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Mass distribution (log scale)
ax = axes[0]
log_mass = np.log10(planets_only[:, MASS])
ax.hist(log_mass, bins=30, edgecolor='black', alpha=0.7, density=True)
ax.axvline(np.log10(1e-7), color='r', ls='--', label='Bounds')
ax.axvline(np.log10(3e-2), color='r', ls='--')
ax.axvline(np.log10(1.7e-4), color='orange', ls='-', lw=2, label=r'$q_{br}$ = 1.7×10⁻⁴')
ax.set_xlabel("log₁₀(Mass/M☉)")
ax.set_ylabel("Density")
ax.set_title("Planet Mass Distribution (log-uniform)")
ax.legend()

# Semi-major axis distribution (log scale)
ax = axes[1]
log_a = np.log10(planets_only[:, SEMI_A])
ax.hist(log_a, bins=30, edgecolor='black', alpha=0.7, density=True)
ax.axvline(np.log10(0.3), color='r', ls='--', label='Bounds')
ax.axvline(np.log10(30), color='r', ls='--')
ax.set_xlabel("log₁₀(a/AU)")
ax.set_ylabel("Density")
ax.set_title("Semi-major Axis Distribution (log-uniform)")
ax.legend()

plt.tight_layout()
plt.show()

## Period Ratio Check

For 2-planet systems, the period ratio must be > 1.3 for stability.

In [ ]:
# Find 2-planet systems
two_planet_mask = (n_planets_per_system == 2)
two_planet_systems = data_3d[two_planet_mask]

a1 = two_planet_systems[:, 0, SEMI_A]
a2 = two_planet_systems[:, 1, SEMI_A]

# Period ratio: P2/P1 = (a2/a1)^1.5, always take the larger ratio
period_ratio = np.maximum(a2/a1, a1/a2) ** 1.5

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Histogram
ax = axes[0]
ax.hist(period_ratio, bins=30, edgecolor='black', alpha=0.7)
ax.axvline(1.3, color='r', ls='--', lw=2, label='Min ratio = 1.3')
ax.set_xlabel("Period Ratio (P_outer/P_inner)")
ax.set_ylabel("Count")
ax.set_title(f"Period Ratios for {len(two_planet_systems)} 2-Planet Systems")
ax.legend()

# Check violations
violations = period_ratio < 1.3
print(f"Period ratio violations (< 1.3): {violations.sum()} ({100*violations.mean():.1f}%)")
print(f"Min period ratio: {period_ratio.min():.3f}")
print(f"Mean period ratio: {period_ratio.mean():.2f}")

# Scatter plot
ax = axes[1]
ax.scatter(a1, a2, alpha=0.5, s=10)
ax.plot([0.1, 100], [0.1, 100], 'k--', lw=1, label='a1 = a2')
# Plot 1.3 ratio boundaries
a_vals = np.logspace(-1, 2, 100)
ax.plot(a_vals, a_vals * 1.3**(2/3), 'r--', label='P2/P1 = 1.3')
ax.plot(a_vals, a_vals / 1.3**(2/3), 'r--')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel("a₁ (AU)")
ax.set_ylabel("a₂ (AU)")
ax.set_title("Semi-major Axis Pairs")
ax.legend()
ax.set_xlim(0.2, 50)
ax.set_ylim(0.2, 50)

plt.tight_layout()
plt.show()

## Eccentricity Distribution

Should be half-Gaussian with σ=0.3, truncated at 0.95.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ecc = planets_only[:, ECC]
ax.hist(ecc, bins=30, edgecolor='black', alpha=0.7, density=True, label='Data')

# Expected half-Gaussian
e_vals = np.linspace(0, 0.95, 100)
sigma = 0.3
half_gauss = 2 * stats.norm.pdf(e_vals, 0, sigma) / stats.norm.cdf(0.95/sigma)  # Truncated
ax.plot(e_vals, half_gauss, 'r-', lw=2, label=f'Half-Gaussian (σ={sigma})')

ax.axvline(0.95, color='orange', ls='--', label='Max e = 0.95')
ax.set_xlabel("Eccentricity")
ax.set_ylabel("Density")
ax.set_title("Eccentricity Distribution")
ax.legend()

print(f"Mean eccentricity: {ecc.mean():.3f}")
print(f"Max eccentricity: {ecc.max():.3f}")
plt.tight_layout()
plt.show()

## Inclination Distribution

GULLS convention: I > 900 means relative to binary orbit.
- I = 1000 means coplanar with binary
- Actual inclination = I_binary + (I - 1000)

Our draw: Gaussian centered on 1000 with σ=5°.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

inc = planets_only[:, INC]

# Raw inclination values
ax = axes[0]
ax.hist(inc, bins=30, edgecolor='black', alpha=0.7)
ax.axvline(1000, color='r', ls='--', lw=2, label='I = 1000 (coplanar)')
ax.set_xlabel("Inclination (degrees)")
ax.set_ylabel("Count")
ax.set_title("Raw Inclination Values")
ax.legend()

# Relative to 1000 (GULLS convention)
ax = axes[1]
relative_inc = inc - 1000
ax.hist(relative_inc, bins=30, edgecolor='black', alpha=0.7, density=True, label='Data')

# Expected Gaussian
x = np.linspace(-20, 20, 100)
sigma = 5.0
ax.plot(x, stats.norm.pdf(x, 0, sigma), 'r-', lw=2, label=f'Gaussian (σ={sigma}°)')
ax.set_xlabel("I - 1000 (degrees)")
ax.set_ylabel("Density")
ax.set_title("Inclination Relative to Binary")
ax.legend()

print(f"Mean inclination: {inc.mean():.1f}°")
print(f"Std of (I - 1000): {relative_inc.std():.2f}°")
plt.tight_layout()
plt.show()

## Longitude Distributions

Both should be uniform in [0, 360).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Longitude of perihelion
ax = axes[0]
omega = planets_only[:, OMEGA]
ax.hist(omega, bins=30, edgecolor='black', alpha=0.7, density=True)
ax.axhline(1/360, color='r', ls='--', lw=2, label='Uniform')
ax.set_xlabel("ω (Longitude of Perihelion, degrees)")
ax.set_ylabel("Density")
ax.set_title("Longitude of Perihelion")
ax.set_xlim(0, 360)
ax.legend()

# Longitude of ascending node
ax = axes[1]
Omega = planets_only[:, LONG_NODE]
ax.hist(Omega, bins=30, edgecolor='black', alpha=0.7, density=True)
ax.axhline(1/360, color='r', ls='--', lw=2, label='Uniform')
ax.set_xlabel("Ω (Longitude of Ascending Node, degrees)")
ax.set_ylabel("Density")
ax.set_title("Longitude of Ascending Node")
ax.set_xlim(0, 360)
ax.legend()

# KS test for uniformity
ks_omega = stats.kstest(omega, 'uniform', args=(0, 360))
ks_Omega = stats.kstest(Omega, 'uniform', args=(0, 360))
print(f"KS test for ω uniformity: statistic={ks_omega.statistic:.4f}, p={ks_omega.pvalue:.4f}")
print(f"KS test for Ω uniformity: statistic={ks_Omega.statistic:.4f}, p={ks_Omega.pvalue:.4f}")

plt.tight_layout()
plt.show()

## Summary Statistics

In [ ]:
print("=" * 50)
print("MULTIPLANET GENERATOR VALIDATION SUMMARY")
print("=" * 50)
print(f"\nSystems analyzed: {n_systems}")
print(f"Total planets: {len(planets_only)}")
print(f"Total moons: {len(moons_only)}")
print(f"\nPlanet counts:")
print(f"  0 planets: {(n_planets_per_system == 0).sum()} ({100*(n_planets_per_system == 0).mean():.1f}%)")
print(f"  1 planet:  {(n_planets_per_system == 1).sum()} ({100*(n_planets_per_system == 1).mean():.1f}%)")
print(f"  2 planets: {(n_planets_per_system == 2).sum()} ({100*(n_planets_per_system == 2).mean():.1f}%)")
print(f"  Mean planets/star: {n_planets_per_system.mean():.3f}")
print(f"\nOrbital elements:")
print(f"  Mass range: {planets_only[:, MASS].min():.2e} - {planets_only[:, MASS].max():.2e} M☉")
print(f"  Semi-major axis: {planets_only[:, SEMI_A].min():.2f} - {planets_only[:, SEMI_A].max():.2f} AU")
print(f"  Eccentricity: {ecc.mean():.3f} ± {ecc.std():.3f}")
print(f"  Inclination - 1000: {relative_inc.mean():.2f}° ± {relative_inc.std():.2f}°")
print(f"\nPeriod ratio (2-planet systems):")
print(f"  Min: {period_ratio.min():.2f}")
print(f"  Mean: {period_ratio.mean():.2f}")
print(f"  Violations (< 1.3): {violations.sum()}")

# Multiplanet Distribution Validation

Validates output from `multiplanet_draw_planet_arrays.py`.

## File Format
```
Mass SemimajorAxis Eccentricity Inclination LongitudePerihelion LongitudeAscNode OrbitType
```

Each system has 3 rows (OrbitType 1, 2, 3 = planet, planet, moon). Mass=0 means empty slot.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

## Load Data

In [ ]:
multiplanet_files = sorted(glob.glob('planets/test_multiplanet/*.planets.*'))
print(f"Found {len(multiplanet_files)} files")

if multiplanet_files:
    df = pd.read_csv(multiplanet_files[0], sep=r'\s+', header=0)
    print(f"Columns: {list(df.columns)}")
    print(f"Shape: {df.shape}")
    print(df.head(9))
else:
    print("Run multiplanet_draw_planet_arrays.py first!")

In [ ]:
# Aggregate all files
if multiplanet_files:
    dfs = [pd.read_csv(f, sep=r'\s+', header=0) for f in multiplanet_files]
    df_all = pd.concat(dfs, ignore_index=True)
    n_systems = len(df_all) // 3
    print(f"Total rows: {len(df_all):,}, Systems: {n_systems:,}")
    
    planet1 = df_all[df_all['OrbitType'] == 1]
    planet2 = df_all[df_all['OrbitType'] == 2]
    moons = df_all[df_all['OrbitType'] == 3]
    
    planet1_real = planet1[planet1['Mass'] > 0]
    planet2_real = planet2[planet2['Mass'] > 0]
    moons_real = moons[moons['Mass'] > 0]
    all_planets = pd.concat([planet1_real, planet2_real])
    
    print(f"Planet1: {len(planet1_real):,} ({100*len(planet1_real)/n_systems:.1f}%)")
    print(f"Planet2: {len(planet2_real):,} ({100*len(planet2_real)/n_systems:.1f}%)")
    print(f"Moons: {len(moons_real):,} ({100*len(moons_real)/n_systems:.1f}%)")

## Planet Count per System

In [ ]:
if multiplanet_files:
    masses = df_all['Mass'].values.reshape(-1, 3)
    planets_per_system = np.sum(masses[:, :2] > 0, axis=1)
    moons_per_system = (masses[:, 2] > 0).astype(int)
    
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    for n in [0, 1, 2]:
        count = (planets_per_system == n).sum()
        ax[0].bar(n, count, label=f'{n}: {count:,} ({100*count/n_systems:.1f}%)')
    ax[0].set_xlabel('Planets per System')
    ax[0].set_ylabel('Count')
    ax[0].legend()
    ax[0].set_title(f'Planet Count (mean={planets_per_system.mean():.3f})')
    
    ax[1].bar([0, 1], [n_systems - moons_per_system.sum(), moons_per_system.sum()])
    ax[1].set_xticks([0, 1])
    ax[1].set_xlabel('Has Moon?')
    ax[1].set_title(f'Moon Count ({100*moons_per_system.mean():.1f}%)')
    plt.tight_layout()
    plt.show()

## Mass Distribution

In [ ]:
if multiplanet_files and len(all_planets) > 0:
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].hist(np.log10(all_planets['Mass']), bins=50, edgecolor='k', alpha=0.7)
    ax[0].axvline(np.log10(1e-7), color='r', ls='--', label='Bounds')
    ax[0].axvline(np.log10(3e-2), color='r', ls='--')
    ax[0].set_xlabel(r'log$_{10}$(Mass / M$_\odot$)')
    ax[0].set_title('Planet Mass (Log-Uniform)')
    ax[0].legend()
    
    if len(moons_real) > 0:
        ax[1].hist(np.log10(moons_real['Mass']), bins=30, edgecolor='k', alpha=0.7, color='orange')
        ax[1].set_xlabel(r'log$_{10}$(Mass / M$_\odot$)')
        ax[1].set_title('Moon Mass')
    plt.tight_layout()
    plt.show()

## Semi-Major Axis Distribution

In [ ]:
if multiplanet_files and len(all_planets) > 0:
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].hist(np.log10(all_planets['SemimajorAxis']), bins=50, edgecolor='k', alpha=0.7)
    ax[0].axvline(np.log10(0.3), color='r', ls='--', label='Bounds')
    ax[0].axvline(np.log10(30), color='r', ls='--')
    ax[0].set_xlabel(r'log$_{10}$(a / AU)')
    ax[0].set_title('Planet Semi-Major Axis (Log-Uniform)')
    ax[0].legend()
    
    if len(moons_real) > 0:
        ax[1].hist(np.log10(moons_real['SemimajorAxis']), bins=30, edgecolor='k', alpha=0.7, color='orange')
        ax[1].set_xlabel(r'log$_{10}$(a / AU)')
        ax[1].set_title('Moon Semi-Major Axis')
    plt.tight_layout()
    plt.show()

## Period Ratio Check

For 2-planet systems: $P_2/P_1 = (a_2/a_1)^{3/2} > 1.3$

In [ ]:
if multiplanet_files:
    masses = df_all['Mass'].values.reshape(-1, 3)
    sma = df_all['SemimajorAxis'].values.reshape(-1, 3)
    two_planet = (masses[:, 0] > 0) & (masses[:, 1] > 0)
    
    if two_planet.sum() > 0:
        a1, a2 = sma[two_planet, 0], sma[two_planet, 1]
        period_ratio = np.maximum(a2/a1, a1/a2) ** 1.5
        
        fig, ax = plt.subplots(1, 2, figsize=(12, 4))
        ax[0].hist(period_ratio, bins=50, edgecolor='k', alpha=0.7)
        ax[0].axvline(1.3, color='r', ls='--', lw=2, label='Min (1.3)')
        ax[0].set_xlabel('Period Ratio')
        ax[0].set_title(f'2-Planet Systems (n={two_planet.sum()})')
        ax[0].legend()
        ax[0].set_xlim(1, None)
        
        ax[1].scatter(a1, a2, alpha=0.3, s=5)
        ax[1].set_xscale('log'); ax[1].set_yscale('log')
        ax[1].set_xlabel('a₁ (AU)'); ax[1].set_ylabel('a₂ (AU)')
        a_range = np.logspace(-0.5, 1.5, 100)
        ax[1].plot(a_range, a_range * 1.3**(2/3), 'r--')
        ax[1].plot(a_range, a_range / 1.3**(2/3), 'r--', label='P=1.3')
        ax[1].legend()
        plt.tight_layout()
        plt.show()
        
        print(f"Violations (P<1.3): {(period_ratio < 1.3).sum()}")
        print(f"Min ratio: {period_ratio.min():.3f}")
    else:
        print("No 2-planet systems")

## Eccentricity & Inclination

In [ ]:
if multiplanet_files and len(all_planets) > 0:
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    
    ax[0].hist(all_planets['Eccentricity'], bins=50, edgecolor='k', alpha=0.7)
    ax[0].set_xlabel('Eccentricity')
    ax[0].set_title(f'Eccentricity (mean={all_planets["Eccentricity"].mean():.3f})')
    ax[0].set_xlim(0, 1)
    
    rel_inc = all_planets['Inclination'] - 1000
    ax[1].hist(rel_inc, bins=50, edgecolor='k', alpha=0.7)
    ax[1].axvline(0, color='r', ls='--', label='Coplanar')
    ax[1].set_xlabel('Inclination - 1000 (deg)')
    ax[1].set_title(f'Relative Inclination (std={rel_inc.std():.2f}°)')
    ax[1].legend()
    plt.tight_layout()
    plt.show()

## Summary

In [ ]:
if multiplanet_files:
    print("="*50)
    print("SUMMARY")
    print("="*50)
    print(f"Files: {len(multiplanet_files)}")
    print(f"Systems: {n_systems:,}")
    print(f"\nPlanets per system: {planets_per_system.mean():.3f}")
    print(f"  0 planets: {(planets_per_system==0).sum():,}")
    print(f"  1 planet: {(planets_per_system==1).sum():,}")
    print(f"  2 planets: {(planets_per_system==2).sum():,}")
    print(f"\nMoons: {moons_per_system.sum():,} ({100*moons_per_system.mean():.1f}%)")
    if len(all_planets) > 0:
        print(f"\nMass range: [{all_planets['Mass'].min():.2e}, {all_planets['Mass'].max():.2e}] M_sun")
        print(f"SMA range: [{all_planets['SemimajorAxis'].min():.3f}, {all_planets['SemimajorAxis'].max():.3f}] AU")
        print(f"Eccentricity: mean={all_planets['Eccentricity'].mean():.3f}")